# Modeling & Evaluation

In [24]:
import pandas as pd
import numpy as np

# قراءة ملف البيانات 
df = pd.read_csv('../data/Original/SeoulBikeData.csv', encoding='latin1')

# طباعة الأبعاد للتأكد من قراءة البيانات بنجاح
print("Data loaded successfully! Shape:", df.shape)

Data loaded successfully! Shape: (8760, 14)


### Categorical Variables Encoding (One-Hot Encoding)

In [10]:
# تحويل المتغيرات الفئوية إلى ترميز رقمي
# نستخدم drop_first=True لتجنب التكرار الخطي (Multicollinearity)
categorical_cols = ['Seasons', 'Holiday', 'Functioning Day']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [11]:
# استعراض أسماء الأعمدة الجديدة بعد التحويل للتأكد
print("--- Columns after Encoding ---")
print(df_encoded.columns.tolist())

--- Columns after Encoding ---
['Date', 'Rented Bike Count', 'Hour', 'Temperature(°C)', 'Humidity(%)', 'Wind speed (m/s)', 'Visibility (10m)', 'Dew point temperature(°C)', 'Solar Radiation (MJ/m2)', 'Rainfall(mm)', 'Snowfall (cm)', 'Seasons_Spring', 'Seasons_Summer', 'Seasons_Winter', 'Holiday_No Holiday', 'Functioning Day_Yes']


## Feature and Target Separation

In [12]:
# فصل المتغيرات المستقلة (X) عن المتغير المستهدف (Rented Bike Count)
# استبعاد التاريخ وعدد الإيجارات من مصفوفة العوامل المستقلة
X = df_encoded.drop(columns=['Rented Bike Count', 'Date'])
y = df_encoded['Rented Bike Count']

print(f"Features shape (X): {X.shape}")
print(f"Target shape (y): {y.shape}")

Features shape (X): (8760, 14)
Target shape (y): (8760,)


## Train/Test Data Splitting

In [13]:
from sklearn.model_selection import train_test_split

# تقسيم البيانات بنسبة 80% للتدريب و 20% للاختبار مع تثبيت العشوائية (random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} rows")
print(f"Testing set size: {X_test.shape[0]} rows")

Training set size: 7008 rows
Testing set size: 1752 rows


## Feature Scaling

In [14]:
from sklearn.preprocessing import StandardScaler

# تهيئة وتطبيق الـ StandardScaler لتوحيد مقياس المتغيرات الرقمية
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data scaling completed successfully!")

Data scaling completed successfully!


## Model Training: Linear Regression

In [15]:
from sklearn.linear_model import LinearRegression

# تهيئة وتدريب نموذج الانحدار الخطي على البيانات الموحدة (Scaled Data)
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

print("Linear Regression model trained successfully!")

Linear Regression model trained successfully!


## Model Evaluation: Linear Regression Metrics

In [16]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# التنبؤ باستخدام بيانات الاختبار وتخزين النتائج
y_pred_lr = lr_model.predict(X_test_scaled)

# حساب مقاييس الأداء لنموذج الانحدار الخطي
lr_mae = mean_absolute_error(y_test, y_pred_lr)
lr_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))
lr_r2 = r2_score(y_test, y_pred_lr)

print("--- Linear Regression Performance ---")
print(f"MAE: {lr_mae:.2f}")
print(f"RMSE: {lr_rmse:.2f}")
print(f"R² Score: {lr_r2:.4f}")

--- Linear Regression Performance ---
MAE: 330.39
RMSE: 440.78
R² Score: 0.5337


## Model Training: Random Forest Regressor

In [18]:
from sklearn.ensemble import RandomForestRegressor

# تهيئة وتدريب نموذج Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

print("Random Forest Regressor model trained successfully!")

Random Forest Regressor model trained successfully!


## Model Evaluation: Random Forest Metrics

In [19]:
# التنبؤ باستخدام بيانات الاختبار لنموذج Random Forest
y_pred_rf = rf_model.predict(X_test)

# حساب مقاييس الأداء لنموذج Random Forest
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
rf_r2 = r2_score(y_test, y_pred_rf)

print("--- Random Forest Regressor Performance ---")
print(f"MAE: {rf_mae:.2f}")
print(f"RMSE: {rf_rmse:.2f}")
print(f"R² Score: {rf_r2:.4f}")

--- Random Forest Regressor Performance ---
MAE: 144.72
RMSE: 240.28
R² Score: 0.8614


## Final Comparison Table

In [20]:
# تجميع النتائج في جدول مقارنة احترافي
comparison_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest Regressor'],
    'MAE': [lr_mae, rf_mae],
    'RMSE': [lr_rmse, rf_rmse],
    'R² Score': [lr_r2, rf_r2]
})

print("--- Final Model Performance Comparison ---")
display(comparison_df.set_index('Model'))

--- Final Model Performance Comparison ---


,MAE,RMSE,R² Score
Model,,,
Linear Regression,330.385218,440.781358,0.533685
Random Forest Regressor,144.722637,240.280063,0.861430


In [22]:
import joblib
import os

os.makedirs('../Models', exist_ok=True)

# حفظ النماذج وأداة الـ Scaler ومقاييس الأعمدة
joblib.dump(lr_model, '../Models/linear_regression_model.pkl')
joblib.dump(rf_model, '../Models/random_forest_model.pkl')
joblib.dump(scaler, '../Models/scaler.pkl')
joblib.dump(X.columns.tolist(), '../Models/model_features.pkl')

print("All models and preprocessing objects saved successfully in Models/!")

All models and preprocessing objects saved successfully in Models/!
